<a href="https://colab.research.google.com/github/jaw039/min-viable-eeg/blob/main/Analyze_Selected_Vs_Random_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Analyze Selected vs. Random Performance

In [ ]:
#Compute Statistics

def analyze_selected_vs_random(selected_results, random_results, kappa_full):
    """Comprehensive analysis comparing selected and random performance."""

    analysis = {}

    for budget in sorted(selected_results['budget'].unique()):
        # Get results for this budget
        selected = selected_results[
            (selected_results['budget'] == budget) &
            (selected_results['method'] == 'selected')
        ]['kappa']

        random = random_results[
            (random_results['budget'] == budget) &
            (random_results['method'] == 'random')
        ]['kappa']

        # Basic statistics
        selected_mean = selected.mean()
        selected_std = selected.std()
        random_mean = random.mean()
        random_std = random.std()

        # Improvement
        improvement = selected_mean - random_mean
        relative_improvement = (improvement / random_mean) * 100

        # Statistical test (t-test)
        from scipy import stats
        t_stat, p_value = stats.ttest_ind(selected, random)

        # Cohen's d effect size
        pooled_std = np.sqrt((selected_std**2 + random_std**2) / 2)
        cohens_d = (selected_mean - random_mean) / pooled_std

        # Retention relative to full
        retention_selected = selected_mean / kappa_full
        retention_random = random_mean / kappa_full

        analysis[budget] = {
            'selected_kappa': f"{selected_mean:.3f} ± {selected_std:.3f}",
            'random_kappa': f"{random_mean:.3f} ± {random_std:.3f}",
            'improvement': f"{improvement:.3f} ({relative_improvement:.1f}%)",
            't_statistic': t_stat,
            'p_value': p_value,
            'significant': p_value < 0.05,
            'cohens_d': cohens_d,
            'retention_selected': retention_selected,
            'retention_random': retention_random
        }

    return pd.DataFrame(analysis).T

In [ ]:
#Interpret Results

def interpret_comparison(analysis_df):
    """Generate interpretation text for comparison results."""

    interpretation = []

    for budget, row in analysis_df.iterrows():
        text = f"\nBudget: {budget} channels\n"
        text += f"Selected: {row['selected_kappa']}\n"
        text += f"Random: {row['random_kappa']}\n"
        text += f"Improvement: {row['improvement']}\n"

        if row['significant']:
            text += f"✓ SIGNIFICANT (p={row['p_value']:.4f})\n"
            effect = 'large' if abs(row['cohens_d']) > 0.8 else \
                     'medium' if abs(row['cohens_d']) > 0.5 else 'small'
            text += f"  Effect size: {effect} (d={row['cohens_d']:.2f})\n"
        else:
            text += f"✗ NOT SIGNIFICANT (p={row['p_value']:.4f})\n"

        interpretation.append(text)

    return "\n".join(interpretation)

In [ ]:
#Create Visualization

import matplotlib.pyplot as plt
import seaborn as sns

def plot_selected_vs_random(selected_results, random_results, budgets):
    """Create comparison plot."""

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # Plot 1: Bar plot with error bars
    ax = axes[0, 0]
    x = np.arange(len(budgets))
    width = 0.35

    selected_means = [selected_results[selected_results['budget']==b]['kappa'].mean()
                     for b in budgets]
    selected_stds = [selected_results[selected_results['budget']==b]['kappa'].std()
                     for b in budgets]
    random_means = [random_results[random_results['budget']==b]['kappa'].mean()
                    for b in budgets]
    random_stds = [random_results[random_results['budget']==b]['kappa'].std()
                   for b in budgets]

    ax.bar(x - width/2, selected_means, width, yerr=selected_stds,
           label='Selected', color='blue', alpha=0.7)
    ax.bar(x + width/2, random_means, width, yerr=random_stds,
           label='Random', color='red', alpha=0.7)
    ax.set_xlabel('Channel Budget')
    ax.set_ylabel('Cohen\'s κ')
    ax.set_title('Selected vs. Random Performance')
    ax.set_xticks(x)
    ax.set_xticklabels(budgets)
    ax.legend()

    # Plot 2: Box plot
    ax = axes[0, 1]
    selected_all = []
    random_all = []
    labels = []

    for b in budgets:
        selected_all.extend(selected_results[selected_results['budget']==b]['kappa'])
        random_all.extend(random_results[random_results['budget']==b]['kappa'])
        labels.extend([f'{b}\nSelected'] * len(selected_results[selected_results['budget']==b]))
        labels.extend([f'{b}\nRandom'] * len(random_results[random_results['budget']==b]))

    data = pd.DataFrame({
        'kappa': selected_all + random_all,
        'condition': labels
    })
    sns.boxplot(x='condition', y='kappa', data=data, ax=ax)
    ax.set_xlabel('Budget and Method')
    ax.set_ylabel('Cohen\'s κ')
    ax.set_title('Distribution Comparison')
    ax.tick_params(axis='x', rotation=45)

    # Plot 3: Improvement bar chart
    ax = axes[1, 0]
    improvements = [selected_means[i] - random_means[i] for i in range(len(budgets))]
    ax.bar(budgets, improvements, color='green', alpha=0.7)
    ax.axhline(y=0, color='black', linestyle='--')
    ax.set_xlabel('Channel Budget')
    ax.set_ylabel('Improvement (Selected - Random)')
    ax.set_title('Improvement of Selected vs. Random')

    # Plot 4: Retention plot
    ax = axes[1, 1]
    kappa_full = selected_results[selected_results['budget']==64]['kappa'].mean()
    selected_retention = [m / kappa_full for m in selected_means]
    random_retention = [m / kappa_full for m in random_means]

    ax.plot(budgets, selected_retention, 'b-o', label='Selected', linewidth=2)
    ax.plot(budgets, random_retention, 'r--s', label='Random', linewidth=2)
    ax.axhline(y=0.9, color='black', linestyle=':', label='90% Threshold')
    ax.set_xlabel('Channel Budget')
    ax.set_ylabel('Retention of Full Performance')
    ax.set_title('Performance Retention')
    ax.legend()

    plt.tight_layout()
    plt.savefig('selected_vs_random_analysis.png', dpi=300)
    plt.show()